In [ ]:
# =========================================================
# Super-Resolution on Kaggle (P100-compatible)
# Role: ML Engineer + Computer Vision Specialist
# Objective: LR -> HR with Bayesian optimization + ablations
# =========================================================

# Notebook flow:
# 1) Environment and dependency setup
# 2) Dataset with augmentations/degradations
# 3) Efficient SR model (FastEDSR)
# 4) Bayesian hyperparameter optimization (Optuna)
# 5) Ablation studies
# 6) Final training: 200 epochs, patience 20, dynamic LR


In [ ]:
# Kaggle-safe dependency install (skip if already available)
# %pip -q install optuna tqdm albumentations --upgrade

In [ ]:
# Force a GPU-compatible PyTorch build for Kaggle GPUs (P100/T4/V100).
# Run this cell once, then RESTART the kernel before continuing.
# %pip install torch==2.3.1 torchvision==0.18.1 torchaudio==2.3.1 \
# --extra-index-url https://download.pytorch.org/whl/cu118

In [ ]:
!nvidia-smi

In [ ]:
# !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118 --force-reinstall

In [ ]:
# !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121 --force-reinstall

In [ ]:
import torch
print(f"Is CUDA available? {torch.cuda.is_available()}")
print(f"Current Device: {torch.cuda.get_device_name(0)}")
# Test a small calculation on GPU
x = torch.randn(1).to("cuda")
print("Success! GPU is talking back.")

In [ ]:
import os
import random
import gc
import json
import math
import time
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import cv2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms

import albumentations as A
from tqdm.auto import tqdm
import optuna

# Reproducibility
def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def resolve_gpu_device_strict():
    if not torch.cuda.is_available():
        raise RuntimeError(
            "CUDA is not available. Use GPU runtime and rerun the PyTorch cu118 install cell, then restart kernel."
        )

    name = torch.cuda.get_device_name(0)
    cap = torch.cuda.get_device_capability(0)
    print("GPU:", name, "| CUDA capability:", cap)

    # Runtime kernel probe: fail fast if installed binaries cannot execute on this GPU.
    try:
        x = torch.randn(1, 3, 8, 8, device="cuda")
        conv = nn.Conv2d(3, 3, 3, 1, 1).to("cuda")
        _ = conv(x)
        torch.cuda.synchronize()
    except Exception as e:
        raise RuntimeError(
            "CUDA runtime is incompatible with the installed PyTorch build. "
            "Run the cu118 install cell, restart kernel, and run again. "
            f"Original error: {str(e).splitlines()[0]}"
        ) from e

    return torch.device("cuda")


seed_everything(42)

device = resolve_gpu_device_strict()
print("Torch:", torch.__version__)
print("Device:", device)

In [ ]:
# --------------------------
# Config
# --------------------------
@dataclass
class CFG:
    # Keep both fields for backward compatibility with older cells.
    image_dir: str = "//kaggle/input/datasets/joe1995/div2k-dataset/DIV2K_train_HR/DIV2K_train_HR"
    train_image_dir: str = "//kaggle/input/datasets/joe1995/div2k-dataset/DIV2K_train_HR/DIV2K_train_HR"
    val_image_dir: str = "//kaggle/input/datasets/joe1995/div2k-dataset/DIV2K_valid_HR/DIV2K_valid_HR"

    save_dir: str = "/kaggle/working/sr_outputs"

    scale: int = 4
    hr_size: int = 128  # HR patch size for training
    batch_size: int = 16
    num_workers: int = 0

    # Number of random patch samples per source image each epoch.
    train_repeat: int = 32

    # Search settings
    n_trials: int = 12
    search_epochs: int = 12

    # Teacher / distillation settings
    teacher_epochs: int = 30
    distill_alpha: float = 0.25
    teacher_warm_start: bool = True

    # Final training settings
    final_epochs: int = 200
    patience: int = 20

cfg = CFG()
os.makedirs(cfg.save_dir, exist_ok=True)
print(cfg)

In [ ]:
# --------------------------
# DIV2K datasets (correct x4 SR protocol)
# --------------------------
def _list_image_paths(image_dir):
    valid_ext = (".png", ".jpg", ".jpeg", ".bmp", ".webp")
    paths = [
        os.path.join(image_dir, x)
        for x in os.listdir(image_dir)
        if x.lower().endswith(valid_ext)
    ]
    paths.sort()
    if len(paths) == 0:
        raise ValueError(f"No images found in: {image_dir}")
    return paths


def _imread_rgb(path):
    bgr = cv2.imread(path, cv2.IMREAD_COLOR)
    if bgr is None:
        raise ValueError(f"Failed to read image: {path}")
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)


def _modcrop(img, scale):
    h, w = img.shape[:2]
    h = h - (h % scale)
    w = w - (w % scale)
    return img[:h, :w]


class SRDataset(Dataset):
    def __init__(self, image_dir, hr_size=128, scale=4, use_augmentation=True, repeat=32):
        self.image_paths = _list_image_paths(image_dir)
        self.hr_size = hr_size
        self.lr_size = hr_size // scale
        self.scale = scale
        self.use_augmentation = use_augmentation
        self.repeat = repeat

        if hr_size % scale != 0:
            raise ValueError("hr_size must be divisible by scale.")

    def __len__(self):
        return len(self.image_paths) * self.repeat

    def _random_crop(self, img):
        h, w = img.shape[:2]
        if h < self.hr_size or w < self.hr_size:
            new_h = max(h, self.hr_size)
            new_w = max(w, self.hr_size)
            img = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_CUBIC)
            h, w = img.shape[:2]

        y = random.randint(0, h - self.hr_size)
        x = random.randint(0, w - self.hr_size)
        return img[y:y + self.hr_size, x:x + self.hr_size]

    def _augment(self, hr):
        if not self.use_augmentation:
            return hr

        if random.random() < 0.5:
            hr = np.ascontiguousarray(np.flip(hr, axis=1))
        if random.random() < 0.5:
            hr = np.ascontiguousarray(np.flip(hr, axis=0))

        k = random.randint(0, 3)
        if k:
            hr = np.ascontiguousarray(np.rot90(hr, k))

        return hr

    def __getitem__(self, idx):
        # idx is expanded by repeat; map back to source image list.
        path = self.image_paths[idx % len(self.image_paths)]
        img = _imread_rgb(path)

        hr = self._random_crop(img)
        hr = self._augment(hr)

        lr = cv2.resize(
            hr,
            (self.lr_size, self.lr_size),
            interpolation=cv2.INTER_CUBIC,
        )

        hr_t = torch.from_numpy(hr).permute(2, 0, 1).float() / 255.0
        lr_t = torch.from_numpy(lr).permute(2, 0, 1).float() / 255.0
        return lr_t, hr_t


class SRValDataset(Dataset):
    def __init__(self, image_dir, scale=4):
        self.image_paths = _list_image_paths(image_dir)
        self.scale = scale

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = _imread_rgb(self.image_paths[idx])
        hr = _modcrop(img, self.scale)

        h, w = hr.shape[:2]
        lr = cv2.resize(
            hr,
            (w // self.scale, h // self.scale),
            interpolation=cv2.INTER_CUBIC,
        )

        hr_t = torch.from_numpy(hr).permute(2, 0, 1).float() / 255.0
        lr_t = torch.from_numpy(lr).permute(2, 0, 1).float() / 255.0
        return lr_t, hr_t

In [ ]:
# --------------------------
# Fast EDSR model (LR -> HR) + cheap attention + EDSR baseline
# --------------------------
class ECALayer(nn.Module):
    def __init__(self, channels, k_size=3):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.conv = nn.Conv1d(1, 1, kernel_size=k_size, padding=(k_size - 1) // 2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        y = self.avg_pool(x)  # [B, C, 1, 1]
        y = y.squeeze(-1).transpose(-1, -2)  # [B, 1, C]
        y = self.conv(y)
        y = self.sigmoid(y)
        y = y.transpose(-1, -2).unsqueeze(-1)  # [B, C, 1, 1]
        return x * y


class SELayer(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        hidden = max(channels // reduction, 8)
        self.net = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(channels, hidden, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden, channels, kernel_size=1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return x * self.net(x)


class LiteResBlock(nn.Module):
    def __init__(self, n_feats, res_scale=0.1, attn_type="eca"):
        super().__init__()
        self.dw = nn.Conv2d(n_feats, n_feats, 3, 1, 1, groups=n_feats)
        self.pw = nn.Conv2d(n_feats, n_feats, 1, 1, 0)
        self.act = nn.ReLU(inplace=True)
        self.res_scale = res_scale
        if attn_type == "eca":
            self.attn = ECALayer(n_feats, k_size=3)
        elif attn_type == "se":
            self.attn = SELayer(n_feats, reduction=8)
        else:
            self.attn = nn.Identity()

    def forward(self, x):
        res = self.act(self.dw(x))
        res = self.pw(res)
        res = self.attn(res)
        return x + res * self.res_scale


class Upsampler(nn.Sequential):
    def __init__(self, scale, n_feats):
        layers = []
        if scale in [2, 4, 8]:
            for _ in range(int(np.log2(scale))):
                layers += [nn.Conv2d(n_feats, 4 * n_feats, 3, 1, 1), nn.PixelShuffle(2)]
        elif scale == 3:
            layers += [nn.Conv2d(n_feats, 9 * n_feats, 3, 1, 1), nn.PixelShuffle(3)]
        else:
            raise ValueError(f"Unsupported scale {scale}")
        super().__init__(*layers)


class FastEDSR(nn.Module):
    def __init__(self, scale=4, n_resblocks=16, n_feats=64, n_colors=3, res_scale=0.1, attn_type="eca"):
        super().__init__()
        self.head = nn.Conv2d(n_colors, n_feats, 3, 1, 1)
        self.body = nn.Sequential(*[LiteResBlock(n_feats, res_scale, attn_type=attn_type) for _ in range(n_resblocks)])
        self.body_tail = nn.Conv2d(n_feats, n_feats, 3, 1, 1)
        self.tail = nn.Sequential(
            Upsampler(scale, n_feats),
            nn.Conv2d(n_feats, n_colors, 3, 1, 1),
        )

    def forward(self, x):
        x = self.head(x)
        res = self.body(x)
        res = self.body_tail(res)
        x = x + res
        x = self.tail(x)
        return x


class EDSRResBlock(nn.Module):
    def __init__(self, n_feats=256, res_scale=0.1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(n_feats, n_feats, 3, 1, 1),
            nn.ReLU(inplace=True),
            nn.Conv2d(n_feats, n_feats, 3, 1, 1),
        )
        self.res_scale = res_scale

    def forward(self, x):
        return x + self.block(x) * self.res_scale


class EDSR(nn.Module):
    # Canonical EDSR design with configurable width/depth for fair baseline comparison.
    def __init__(self, scale=4, n_resblocks=16, n_feats=96, n_colors=3, res_scale=0.1):
        super().__init__()
        self.head = nn.Conv2d(n_colors, n_feats, 3, 1, 1)
        self.body = nn.Sequential(*[EDSRResBlock(n_feats, res_scale=res_scale) for _ in range(n_resblocks)])
        self.body_tail = nn.Conv2d(n_feats, n_feats, 3, 1, 1)
        self.tail = nn.Sequential(
            Upsampler(scale, n_feats),
            nn.Conv2d(n_feats, n_colors, 3, 1, 1),
        )

    def forward(self, x):
        x = self.head(x)
        res = self.body(x)
        res = self.body_tail(res)
        x = x + res
        x = self.tail(x)
        return x


def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

In [ ]:
# --------------------------
# Training / Eval Utilities
# --------------------------
def rgb_to_y(x):
    # x in [0, 1], shape [N, 3, H, W]
    r = x[:, 0:1, :, :]
    g = x[:, 1:2, :, :]
    b = x[:, 2:3, :, :]
    # MATLAB-style Y conversion used by many SR benchmarks.
    y = (16.0 / 255.0) + (65.481 * r + 128.553 * g + 24.966 * b) / 255.0
    return y


def psnr_y_shave(sr, hr, scale=4, eps=1e-12):
    sr = sr.float().clamp(0, 1)
    hr = hr.float().clamp(0, 1)

    sr_y = rgb_to_y(sr)
    hr_y = rgb_to_y(hr)

    if scale > 0 and sr_y.shape[-1] > 2 * scale and sr_y.shape[-2] > 2 * scale:
        sr_y = sr_y[:, :, scale:-scale, scale:-scale]
        hr_y = hr_y[:, :, scale:-scale, scale:-scale]

    mse = F.mse_loss(sr_y, hr_y, reduction="none").flatten(1).mean(dim=1)
    psnr = 10.0 * torch.log10(1.0 / torch.clamp(mse, min=eps))
    return psnr.mean().item()


def make_grad_scaler(device):
    return torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))


class EarlyStopping:
    def __init__(self, patience=20, mode="max", min_delta=1e-4):
        self.patience = patience
        self.mode = mode
        self.min_delta = min_delta
        self.best = None
        self.counter = 0

    def step(self, metric):
        if self.best is None:
            self.best = metric
            return False

        improved = (metric > self.best + self.min_delta) if self.mode == "max" else (metric < self.best - self.min_delta)
        if improved:
            self.best = metric
            self.counter = 0
            return False

        self.counter += 1
        return self.counter >= self.patience


def train_one_epoch(model, loader, optimizer, criterion, device, scaler=None, use_amp=True):
    model.train()
    running_loss = 0.0
    pbar = tqdm(loader, desc="train", leave=False)
    for lr, hr in pbar:
        lr = lr.to(device, non_blocking=True)
        hr = hr.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        if use_amp and device.type == "cuda":
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                sr = model(lr)
                loss = criterion(sr, hr)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            sr = model(lr)
            loss = criterion(sr, hr)
            loss.backward()
            optimizer.step()

        running_loss += loss.item()
        pbar.set_postfix(loss=f"{loss.item():.4f}")

    return running_loss / max(len(loader), 1)


def train_one_epoch_distill(student, teacher, loader, optimizer, criterion, device, alpha=0.25, scaler=None, use_amp=True):
    student.train()
    teacher.eval()
    running_loss = 0.0
    pbar = tqdm(loader, desc="distill", leave=False)
    for lr, hr in pbar:
        lr = lr.to(device, non_blocking=True)
        hr = hr.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        if use_amp and device.type == "cuda":
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                with torch.no_grad():
                    t_sr = teacher(lr).detach()
                s_sr = student(lr)
                hard_loss = criterion(s_sr, hr)
                soft_loss = F.l1_loss(s_sr, t_sr)
                loss = (1.0 - alpha) * hard_loss + alpha * soft_loss
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            with torch.no_grad():
                t_sr = teacher(lr).detach()
            s_sr = student(lr)
            hard_loss = criterion(s_sr, hr)
            soft_loss = F.l1_loss(s_sr, t_sr)
            loss = (1.0 - alpha) * hard_loss + alpha * soft_loss
            loss.backward()
            optimizer.step()

        running_loss += loss.item()
        pbar.set_postfix(loss=f"{loss.item():.4f}")

    return running_loss / max(len(loader), 1)


@torch.inference_mode()
def evaluate(model, loader, criterion, device, use_amp=True):
    model.eval()
    total_loss = 0.0
    total_psnr = 0.0
    pbar = tqdm(loader, desc="valid", leave=False)
    for lr, hr in pbar:
        lr = lr.to(device, non_blocking=True)
        hr = hr.to(device, non_blocking=True)

        if use_amp and device.type == "cuda":
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                sr = model(lr)
                loss = criterion(sr, hr)
        else:
            sr = model(lr)
            loss = criterion(sr, hr)

        sr = sr.float().clamp(0, 1)
        total_loss += loss.item()
        total_psnr += psnr_y_shave(sr, hr.float(), scale=cfg.scale)

    avg_loss = total_loss / max(len(loader), 1)
    avg_psnr = total_psnr / max(len(loader), 1)
    return avg_loss, avg_psnr


@torch.inference_mode()
def benchmark_latency_ms(model, sample_lr, device, warmup=20, iters=80):
    model.eval()
    x = sample_lr.to(device)
    if device.type == "cuda":
        for _ in range(warmup):
            _ = model(x)
        torch.cuda.synchronize()
        start = torch.cuda.Event(enable_timing=True)
        end = torch.cuda.Event(enable_timing=True)
        start.record()
        for _ in range(iters):
            _ = model(x)
        end.record()
        torch.cuda.synchronize()
        return start.elapsed_time(end) / iters
    else:
        for _ in range(warmup):
            _ = model(x)
        t0 = time.perf_counter()
        for _ in range(iters):
            _ = model(x)
        t1 = time.perf_counter()
        return (t1 - t0) * 1000.0 / iters


def save_model(path, model, meta):
    torch.save({
        "model_state_dict": model.state_dict(),
        "meta": meta,
    }, path)

In [ ]:
# --------------------------
# Data preparation + sample visualization
# --------------------------
train_ds = SRDataset(
    image_dir=cfg.train_image_dir,
    hr_size=cfg.hr_size,
    scale=cfg.scale,
    use_augmentation=True,
    repeat=cfg.train_repeat,
)

val_ds = SRValDataset(
    image_dir=cfg.val_image_dir,
    scale=cfg.scale,
)

# Keep DataLoader single-process in notebooks to avoid multiprocessing cleanup errors.
train_loader = DataLoader(
    train_ds,
    batch_size=cfg.batch_size,
    shuffle=True,
    num_workers=cfg.num_workers,
    pin_memory=(device.type == "cuda"),
    persistent_workers=False,
    drop_last=True,
    generator=torch.Generator().manual_seed(42),
)

val_loader = DataLoader(
    val_ds,
    batch_size=1,  # full-image validation for stable benchmark PSNR
    shuffle=False,
    num_workers=cfg.num_workers,
    pin_memory=(device.type == "cuda"),
    persistent_workers=False,
    drop_last=False,
)

print(f"Train source images: {len(train_ds.image_paths)} | Train samples/epoch: {len(train_ds)}")
print(f"Validation images: {len(val_ds)}")

lr_ex, hr_ex = train_ds[0]
fig, ax = plt.subplots(1, 2, figsize=(8, 4))
ax[0].imshow(lr_ex.permute(1, 2, 0).numpy()); ax[0].set_title("Train LR patch")
ax[1].imshow(hr_ex.permute(1, 2, 0).numpy()); ax[1].set_title("Train HR patch")
for a in ax:
    a.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# --------------------------
# Bayesian Optimization (Optuna)
# --------------------------

class CharbonnierLoss(nn.Module):
    def __init__(self, eps=1e-6):
        super().__init__()
        self.eps = eps

    def forward(self, x, y):
        return torch.mean(torch.sqrt((x - y) ** 2 + self.eps))


def build_model_from_trial(trial):
    trial_group = trial.suggest_categorical("trial_group", ["fast", "quality_plus"] )

    if trial_group == "quality_plus":
        n_resblocks = trial.suggest_int("qp_n_resblocks", 12, 20, step=4)
        n_feats = trial.suggest_categorical("qp_n_feats", [64, 80, 96])
    else:
        n_resblocks = trial.suggest_int("fast_n_resblocks", 8, 16, step=4)
        n_feats = trial.suggest_categorical("fast_n_feats", [48, 64, 80])

    res_scale = trial.suggest_float("res_scale", 0.05, 0.2)
    attn_type = trial.suggest_categorical("attn_type", ["eca", "se", "none"] )

    return FastEDSR(
        scale=cfg.scale,
        n_resblocks=n_resblocks,
        n_feats=n_feats,
        res_scale=res_scale,
        attn_type=attn_type,
    ).to(device)


def objective(trial):
    lr_rate = trial.suggest_float("lr", 1e-4, 5e-3, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-8, 1e-4, log=True)
    loss_name = trial.suggest_categorical("loss", ["l1", "charbonnier"] )

    model = build_model_from_trial(trial)

    criterion = nn.L1Loss() if loss_name == "l1" else CharbonnierLoss()

    optimizer = torch.optim.AdamW(
        model.parameters(), lr=lr_rate, weight_decay=weight_decay
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=0.5, patience=4, min_lr=1e-6
    )

    scaler = make_grad_scaler(device)
    early_stop = EarlyStopping(patience=6, mode="max")

    best_psnr = -1.0

    for epoch in range(cfg.search_epochs):
        _ = train_one_epoch(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            criterion=criterion,
            device=device,
            scaler=scaler,
            use_amp=True,
        )

        _, val_psnr = evaluate(
            model, val_loader, criterion, device, use_amp=True
        )

        val_psnr_value = val_psnr.item() if torch.is_tensor(val_psnr) else val_psnr

        scheduler.step(val_psnr_value)

        trial.report(val_psnr_value, step=epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()

        best_psnr = max(best_psnr, val_psnr_value)

        if early_stop.step(val_psnr_value):
            break

    del model
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()

    return best_psnr


# --------------------------
# Run study / use finalized best values
# --------------------------

sampler = optuna.samplers.TPESampler(seed=42)
pruner = optuna.pruners.MedianPruner(
    n_startup_trials=4, n_warmup_steps=3
)

study = optuna.create_study(
    direction="maximize", sampler=sampler, pruner=pruner
)

# Finalized best values from your completed tuning run.
best_trial_value = 18.285327529907228
best_p = {
    "lr": 0.0010150667045928574,
    "weight_decay": 1.5339162591163588e-08,
    "loss": "l1",
    "trial_group": "quality_plus",
    "qp_n_resblocks": 20,
    "qp_n_feats": 64,
    "res_scale": 0.15263495397682356,
    "attn_type": "none",
}

print("Best trial value (PSNR):", best_trial_value)
print("Best params:", best_p)

best_params_path = os.path.join(cfg.save_dir, "best_optuna_params.json")
with open(best_params_path, "w", encoding="utf-8") as f:
    json.dump(best_p, f, indent=2)

print("Saved best params to", best_params_path)

In [ ]:
# --------------------------
# Ablation study
# --------------------------

class CharbonnierLoss(nn.Module):
    def __init__(self, eps=1e-6):
        super().__init__()
        self.eps = eps

    def forward(self, x, y):
        return torch.mean(torch.sqrt((x - y) ** 2 + self.eps))


def make_loaders(use_aug=True, batch_size=None):
    tr_ds = SRDataset(
        cfg.train_image_dir,
        hr_size=cfg.hr_size,
        scale=cfg.scale,
        use_augmentation=use_aug,
        repeat=cfg.train_repeat,
    )

    vl_ds = SRValDataset(
        cfg.val_image_dir,
        scale=cfg.scale,
    )

    bs = batch_size or cfg.batch_size

    tr_loader = DataLoader(
        tr_ds,
        batch_size=bs,
        shuffle=True,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        persistent_workers=False,
        drop_last=True,
    )

    vl_loader = DataLoader(
        vl_ds,
        batch_size=1,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        persistent_workers=False,
        drop_last=False,
    )

    return tr_loader, vl_loader


def run_ablation(name, use_aug, use_scheduler, epochs=10):
    p = best_p

    n_resblocks = p.get("qp_n_resblocks", p.get("fast_n_resblocks"))
    n_feats = p.get("qp_n_feats", p.get("fast_n_feats"))

    model = FastEDSR(
        scale=cfg.scale,
        n_resblocks=int(n_resblocks),
        n_feats=int(n_feats),
        res_scale=float(p["res_scale"]),
        attn_type=str(p.get("attn_type", "eca")),
    ).to(device)

    criterion = nn.L1Loss() if p["loss"] == "l1" else CharbonnierLoss()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=float(p["lr"]),
        weight_decay=float(p["weight_decay"]),
    )

    scheduler = (
        torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode="max",
            factor=0.5,
            patience=3,
            min_lr=1e-6,
        )
        if use_scheduler else None
    )

    scaler = make_grad_scaler(device)
    tr_loader, vl_loader = make_loaders(use_aug=use_aug)

    best_psnr = -1.0

    for _ in tqdm(range(epochs), desc=f"ablation:{name}"):
        _ = train_one_epoch(
            model,
            tr_loader,
            optimizer,
            criterion,
            device,
            scaler=scaler,
            use_amp=True,
        )

        _, val_psnr = evaluate(
            model,
            vl_loader,
            criterion,
            device,
            use_amp=True,
        )

        val_psnr_value = val_psnr.item() if torch.is_tensor(val_psnr) else val_psnr

        best_psnr = max(best_psnr, val_psnr_value)

        if scheduler is not None:
            scheduler.step(val_psnr_value)

    del tr_loader, vl_loader, model
    gc.collect()

    if device.type == "cuda":
        torch.cuda.empty_cache()

    return best_psnr


# --------------------------
# Run ablations
# --------------------------

ablation_configs = [
    {"name": "baseline_no_aug_no_sched", "use_aug": False, "use_scheduler": False},
    {"name": "aug_only", "use_aug": True, "use_scheduler": False},
    {"name": "aug_plus_scheduler", "use_aug": True, "use_scheduler": True},
]

ablation_rows = []

for ab in ablation_configs:
    score = run_ablation(**ab, epochs=8)
    ablation_rows.append({**ab, "best_psnr": score})


ablation_df = (
    pd.DataFrame(ablation_rows)
    .sort_values("best_psnr", ascending=False)
    .reset_index(drop=True)
)

display(ablation_df)


ablation_path = os.path.join(cfg.save_dir, "ablation_results.csv")
ablation_df.to_csv(ablation_path, index=False)

print("Saved ablation results to", ablation_path)

In [ ]:
# --------------------------
# Teacher EDSR training + student distillation final training
# --------------------------

class CharbonnierLoss(nn.Module):
    def __init__(self, eps=1e-6):
        super().__init__()
        self.eps = eps

    def forward(self, x, y):
        return torch.mean(torch.sqrt((x - y) ** 2 + self.eps))


n_resblocks = best_p.get("qp_n_resblocks", best_p.get("fast_n_resblocks"))
n_feats = best_p.get("qp_n_feats", best_p.get("fast_n_feats"))

# --------------------------
# Teacher model
# --------------------------

teacher_model = EDSR(
    scale=cfg.scale,
    n_resblocks=int(n_resblocks),
    n_feats=max(int(n_feats), 96),
    res_scale=float(best_p["res_scale"]),
).to(device)

criterion = nn.L1Loss() if best_p["loss"] == "l1" else CharbonnierLoss()

teacher_opt = torch.optim.AdamW(
    teacher_model.parameters(),
    lr=float(best_p["lr"]),
    weight_decay=float(best_p["weight_decay"]),
)

teacher_sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
    teacher_opt, mode="max", factor=0.5, patience=5, min_lr=1e-6
)

teacher_scaler = make_grad_scaler(device)
teacher_es = EarlyStopping(patience=max(cfg.patience // 2, 8), mode="max")

teacher_best_psnr = -1.0
teacher_best_path = os.path.join(cfg.save_dir, "best_teacher_edsr.pth")

if cfg.teacher_warm_start:
    for epoch in tqdm(range(1, cfg.teacher_epochs + 1), desc="teacher-train"):
        tr_loss = train_one_epoch(
            teacher_model, train_loader, teacher_opt,
            criterion, device, scaler=teacher_scaler, use_amp=True
        )

        vl_loss, vl_psnr = evaluate(
            teacher_model, val_loader, criterion, device, use_amp=True
        )

        vl_psnr_val = vl_psnr.item() if torch.is_tensor(vl_psnr) else vl_psnr

        teacher_sched.step(vl_psnr_val)

        if vl_psnr_val > teacher_best_psnr:
            teacher_best_psnr = vl_psnr_val
            save_model(
                teacher_best_path,
                teacher_model,
                meta={
                    "best_psnr": teacher_best_psnr,
                    "cfg": cfg.__dict__,
                    "source": "teacher_edsr",
                },
            )

        if epoch % 10 == 0 or epoch == 1:
            print(
                f"Teacher Epoch {epoch:03d} | "
                f"train_loss={tr_loss:.4f} | "
                f"val_loss={vl_loss:.4f} | "
                f"val_psnr={vl_psnr_val:.3f}"
            )

        if teacher_es.step(vl_psnr_val):
            print(f"Teacher early stopping at epoch {epoch}.")
            break


# --------------------------
# Load best teacher
# --------------------------

if os.path.exists(teacher_best_path):
    t_ckpt = torch.load(teacher_best_path, map_location=device)
    teacher_model.load_state_dict(t_ckpt["model_state_dict"] )

teacher_model.eval()
for p in teacher_model.parameters():
    p.requires_grad_(False)


# --------------------------
# Student model
# --------------------------

student_model = FastEDSR(
    scale=cfg.scale,
    n_resblocks=int(n_resblocks),
    n_feats=int(n_feats),
    res_scale=float(best_p["res_scale"]),
    attn_type=str(best_p.get("attn_type", "eca")),
).to(device)

student_opt = torch.optim.AdamW(
    student_model.parameters(),
    lr=float(best_p["lr"]),
    weight_decay=float(best_p["weight_decay"]),
)

student_sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
    student_opt, mode="max", factor=0.5, patience=6, min_lr=1e-6
)

student_scaler = make_grad_scaler(device)
student_es = EarlyStopping(patience=cfg.patience, mode="max")

history = []
best_psnr = -1.0
best_path = os.path.join(cfg.save_dir, "best_sr_model.pth")


# --------------------------
# Distillation training
# --------------------------

for epoch in tqdm(range(1, cfg.final_epochs + 1), desc="student-distill-train"):

    train_loss = train_one_epoch_distill(
        student=student_model,
        teacher=teacher_model,
        loader=train_loader,
        optimizer=student_opt,
        criterion=criterion,
        device=device,
        alpha=cfg.distill_alpha,
        scaler=student_scaler,
        use_amp=True,
    )

    val_loss, val_psnr = evaluate(
        student_model, val_loader, criterion, device, use_amp=True
    )

    val_psnr_val = val_psnr.item() if torch.is_tensor(val_psnr) else val_psnr

    student_sched.step(val_psnr_val)

    current_lr = student_opt.param_groups[0]["lr"]

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_psnr": val_psnr_val,
        "lr": current_lr,
    })

    if val_psnr_val > best_psnr:
        best_psnr = val_psnr_val
        save_model(
            best_path,
            student_model,
            meta={
                "best_psnr": best_psnr,
                "best_params": best_p,
                "distill_alpha": cfg.distill_alpha,
                "cfg": cfg.__dict__,
            },
        )

    if epoch % 10 == 0 or epoch == 1:
        print(
            f"Student Epoch {epoch:03d} | "
            f"train_loss={train_loss:.4f} | "
            f"val_loss={val_loss:.4f} | "
            f"val_psnr={val_psnr_val:.3f} | "
            f"lr={current_lr:.2e}"
        )

    if student_es.step(val_psnr_val):
        print(f"Student early stopping at epoch {epoch} (patience={cfg.patience}).")
        break


# --------------------------
# Save history
# --------------------------

hist_df = pd.DataFrame(history)
hist_path = os.path.join(cfg.save_dir, "training_history.csv")
hist_df.to_csv(hist_path, index=False)

print("Student best PSNR:", best_psnr)
print("Teacher best PSNR:", teacher_best_psnr)
print("Best student model:", best_path)
print("Best teacher model:", teacher_best_path)
print("History CSV:", hist_path)

In [ ]:
# --------------------------
# Load best student/teacher, compare speed/params/quality, and visualize outputs
# --------------------------

class CharbonnierLoss(nn.Module):
    def __init__(self, eps=1e-6):
        super().__init__()
        self.eps = eps

    def forward(self, x, y):
        return torch.mean(torch.sqrt((x - y) ** 2 + self.eps))


# ✅ resolve params
n_resblocks = best_p.get("qp_n_resblocks", best_p.get("fast_n_resblocks"))
n_feats = best_p.get("qp_n_feats", best_p.get("fast_n_feats"))

# --------------------------
# Models
# --------------------------

student = FastEDSR(
    scale=cfg.scale,
    n_resblocks=int(n_resblocks),
    n_feats=int(n_feats),
    res_scale=float(best_p["res_scale"]),
    attn_type=str(best_p.get("attn_type", "eca")),
).to(device)

teacher = EDSR(
    scale=cfg.scale,
    n_resblocks=int(n_resblocks),
    n_feats=max(int(n_feats), 96),
    res_scale=float(best_p["res_scale"]),
).to(device)

# --------------------------
# Load weights
# --------------------------

student_ckpt = torch.load(os.path.join(cfg.save_dir, "best_sr_model.pth"), map_location=device)
student.load_state_dict(student_ckpt["model_state_dict"])

teacher_ckpt = torch.load(os.path.join(cfg.save_dir, "best_teacher_edsr.pth"), map_location=device)
teacher.load_state_dict(teacher_ckpt["model_state_dict"])

student.eval()
teacher.eval()

# --------------------------
# Evaluation
# --------------------------

criterion_cmp = nn.L1Loss() if best_p["loss"] == "l1" else CharbonnierLoss()

val_loss_student, val_psnr_student = evaluate(student, val_loader, criterion_cmp, device, use_amp=True)
val_loss_teacher, val_psnr_teacher = evaluate(teacher, val_loader, criterion_cmp, device, use_amp=True)

# ✅ ensure float
val_psnr_student = val_psnr_student.item() if torch.is_tensor(val_psnr_student) else val_psnr_student
val_psnr_teacher = val_psnr_teacher.item() if torch.is_tensor(val_psnr_teacher) else val_psnr_teacher

# --------------------------
# Sample inference
# --------------------------

val_lr, val_hr = next(iter(val_loader))
val_lr = val_lr.to(device)
val_hr = val_hr.to(device)

with torch.inference_mode():
    with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(device.type == "cuda")):
        sr_student = student(val_lr[:1]).float().clamp(0, 1)
        sr_teacher = teacher(val_lr[:1]).float().clamp(0, 1)

# --------------------------
# Benchmark
# --------------------------

lat_student = benchmark_latency_ms(student, val_lr[:1], device=device)
lat_teacher = benchmark_latency_ms(teacher, val_lr[:1], device=device)

params_student = count_params(student)
params_teacher = count_params(teacher)

# --------------------------
# Comparison table
# --------------------------

cmp_df = pd.DataFrame([
    {
        "model": "Student FastEDSR + distillation",
        "val_loss": val_loss_student,
        "val_psnr": val_psnr_student,
        "params_M": params_student / 1e6,
        "latency_ms": lat_student,
    },
    {
        "model": "Teacher EDSR",
        "val_loss": val_loss_teacher,
        "val_psnr": val_psnr_teacher,
        "params_M": params_teacher / 1e6,
        "latency_ms": lat_teacher,
    },
])

cmp_df["speedup_vs_edsr"] = lat_teacher / cmp_df["latency_ms"]

display(cmp_df)

# --------------------------
# Visualization
# --------------------------

lr_np = val_lr[0].detach().cpu().permute(1, 2, 0).numpy()
hr_np = val_hr[0].detach().cpu().permute(1, 2, 0).numpy()
sr_student_np = sr_student[0].detach().cpu().permute(1, 2, 0).numpy()
sr_teacher_np = sr_teacher[0].detach().cpu().permute(1, 2, 0).numpy()

fig, ax = plt.subplots(1, 4, figsize=(16, 4))
ax[0].imshow(lr_np); ax[0].set_title("LR")
ax[1].imshow(sr_student_np); ax[1].set_title("Student SR (FastEDSR)")
ax[2].imshow(sr_teacher_np); ax[2].set_title("Teacher SR (EDSR)")
ax[3].imshow(hr_np); ax[3].set_title("HR Target")

for a in ax:
    a.axis("off")

plt.tight_layout()
plt.show()

# --------------------------
# Save artifacts
# --------------------------

final_path = os.path.join(cfg.save_dir, "final_sr_model_last.pth")
torch.save(
    {
        "model_state_dict": student.state_dict(),
        "meta": {"cfg": cfg.__dict__, "best_params": best_p},
    },
    final_path,
)

cmp_path = os.path.join(cfg.save_dir, "student_vs_edsr_comparison.csv")
cmp_df.to_csv(cmp_path, index=False)

print("Saved final student model to", final_path)
print("Saved comparison table to", cmp_path)
print("Param reduction vs EDSR (%):", round(100.0 * (1 - params_student / params_teacher), 2))
print("PSNR delta (student - teacher):", round(val_psnr_student - val_psnr_teacher, 4))

In [ ]:
# --------------------------
# Quick summary for resume-ready reporting
# --------------------------
print("Suggested resume lines:")
print("1) Built a super-resolution pipeline (LR->HR) on Kaggle with FastEDSR and mixed precision training.")
print("2) Added Bayesian hyperparameter optimization (Optuna) and ablation studies for data augmentation/scheduler impact.")
print("3) Trained with dynamic LR scheduling + early stopping (patience=20), saved reproducible best/final checkpoints and training logs.")

print("\nAll outputs stored in:", cfg.save_dir)